In [ ]:
# Cell 0: Install dependencies & mount Google Drive
!pip install -q torchvision torch roboflow pyyaml

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 1: Imports
import os
import copy
import time
import random
import zipfile
import shutil
import yaml
import json
from pathlib import Path
from datetime import datetime
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image


import torch
import torchvision
from torchvision import transforms
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn,
    fasterrcnn_resnet50_fpn_v2,
    fasterrcnn_mobilenet_v3_large_fpn,
    FasterRCNN_ResNet50_FPN_Weights,
    FasterRCNN_ResNet50_FPN_V2_Weights,
    FasterRCNN_MobileNet_V3_Large_FPN_Weights,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import Dataset, DataLoader

import torch.backends.cudnn as cudnn
cudnn.benchmark = True

print(f"PyTorch  : {torch.__version__}")
print(f"Torchvision: {torchvision.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

In [ ]:
# Cell 2: Constants / Config
import os, json
from datetime import datetime

HOME = os.getcwd()

# Experiment selector
# Pick ONE:
#   'EXP1_ALL'
#   'EXP2_OCCLUSION'
#   'EXP3_CLUSTER'
#   'EXP4_ILLUMINATION'
#   'EXP5_SCALE'
EXPERIMENT = 'EXP1_ALL'

# Unified split paths (read directly from Drive)
UNIFIED_ROOT = f'/content/drive/MyDrive/Dataset/{EXPERIMENT}/COCO'
TRAIN_DIR    = os.path.join(UNIFIED_ROOT, 'train')
VALID_DIR    = os.path.join(UNIFIED_ROOT, 'valid')
TEST_DIR     = os.path.join(UNIFIED_ROOT, 'test')

# Run config
RUN_STAMP = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
RUN_NAME  = f'{EXPERIMENT}_{RUN_STAMP}'
RUN_DIR   = f'/content/drive/MyDrive/Oranges/FasterRCNN_Runs/{RUN_NAME}'

CONFIG = {
    'epochs'      : 100,
    'batch_size'  : 16, 
    'lr'          : 0.005,
    'momentum'    : 0.9,
    'weight_decay': 0.0005,
    'lr_step_size': 8,
    'lr_gamma'    : 0.1,
    'num_workers' : 4,
    'save_path'   : RUN_DIR,
}

MODEL_DETAILS = {
    'resnet50':   (fasterrcnn_resnet50_fpn,           FasterRCNN_ResNet50_FPN_Weights.DEFAULT,          'FasterRCNN-ResNet50-FPN'),
    'resnet50v2': (fasterrcnn_resnet50_fpn_v2,        FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT,       'FasterRCNN-ResNet50v2-FPN'),
    'mobilenet':  (fasterrcnn_mobilenet_v3_large_fpn, FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT, 'FasterRCNN-MobileNetV3-FPN'),
}

NUM_CLASSES = 2  # 0 = background | 1 = orange

os.makedirs(RUN_DIR, exist_ok=True)
with open(os.path.join(RUN_DIR, 'config.json'), 'w') as f:
    json.dump({'experiment': EXPERIMENT, 'run_stamp': RUN_STAMP,
               'num_classes': NUM_CLASSES, **CONFIG}, f, indent=2)

IMAGE_EXTS = {'.jpg', '.jpeg', '.png'}

print(f'Experiment : {EXPERIMENT}')
print(f'Train dir  : {TRAIN_DIR}')
print(f'Valid dir  : {VALID_DIR}')
print(f'Test dir   : {TEST_DIR}')
print(f'Run dir    : {RUN_DIR}')

In [ ]:
# Cell 2b: Copy split to local Colab storage for faster I/O
import shutil

LOCAL_ROOT = f'/content/unified_split/{EXPERIMENT}/COCO'

if not os.path.exists(LOCAL_ROOT):
    print('Copying split to local storage...')
    shutil.copytree(
        f'/content/drive/MyDrive/Dataset/{EXPERIMENT}/COCO',
        LOCAL_ROOT
    )
    print('Done')
else:
    print('Local copy already exists: skipping')

TRAIN_DIR = os.path.join(LOCAL_ROOT, 'train')
VALID_DIR = os.path.join(LOCAL_ROOT, 'valid')
TEST_DIR  = os.path.join(LOCAL_ROOT, 'test')

print(f'Train dir : {TRAIN_DIR}')
print(f'Valid dir : {VALID_DIR}')
print(f'Test dir  : {TEST_DIR}')

In [ ]:
# Cell 3: Verify unified split
from pathlib import Path

def split_stats(split_dir):
    jp = Path(split_dir) / '_annotations.coco.json'
    with open(jp) as f: d = json.load(f)
    return len(d['images']), len(d['annotations'])

print(f'{"Split":<8} {"Images":>8} {"Oranges":>10}')
for name, d in [('train', TRAIN_DIR), ('valid', VALID_DIR), ('test', TEST_DIR)]:
    imgs, anns = split_stats(d)
    print(f'{name:<8} {imgs:>8} {anns:>10}')

In [ ]:
# Cell 4: OrangeDataset (COCO format, boxes only, image cache + augmentation) + DataLoaders

import torchvision.transforms.functional as TF

TARGET_SIZE = 640

class OrangeDataset(Dataset):
    """
    Reads images + COCO-format _annotations.coco.json and serves them
    in the format expected by torchvision Faster R-CNN:
        image  : FloatTensor [3, H, W], values in [0, 1]
        target : dict with keys 'boxes' (FloatTensor [N,4]) and
                               'labels' (Int64Tensor [N])
    Images are resized to TARGET_SIZE x TARGET_SIZE and cached in RAM.
    Bounding boxes are scaled accordingly.
    Horizontal flip augmentation applied at training time only.
    """

    def __init__(self, split_dir, transforms=None, augment=False):
        self.transforms = transforms
        self.augment    = augment

        with open(Path(split_dir) / '_annotations.coco.json') as f:
            coco = json.load(f)

        self.images = coco['images']

        annot_index = {img['id']: [] for img in self.images}
        for ann in coco['annotations']:
            if ann['category_id'] == 1:
                annot_index[ann['image_id']].append(ann)

        print(f'Pre-loading images for {len(self.images)} images...')
        self.img_cache = {}
        self.box_cache = {}

        for img_info in self.images:
            img_path = Path(split_dir) / img_info['file_name']
            orig_w, orig_h = img_info['width'], img_info['height']
            scale_x = TARGET_SIZE / orig_w
            scale_y = TARGET_SIZE / orig_h

            # Cache image tensor at 640x640
            img   = Image.open(img_path).convert('RGB').resize(
                        (TARGET_SIZE, TARGET_SIZE), Image.BILINEAR)
            img_t = TF.to_tensor(img)
            self.img_cache[img_info['id']] = img_t

            # Cache boxes scaled to 640x640
            boxes, labels = [], []
            for ann in annot_index.get(img_info['id'], []):
                x, y, w, h = [float(v) for v in ann['bbox']]
                x1 = x * scale_x
                y1 = y * scale_y
                x2 = (x + w) * scale_x
                y2 = (y + h) * scale_y
                if x2 <= x1 or y2 <= y1:
                    continue
                boxes.append([x1, y1, x2, y2])
                labels.append(1)

            if boxes:
                boxes_t  = torch.as_tensor(boxes,  dtype=torch.float32)
                labels_t = torch.as_tensor(labels, dtype=torch.int64)
            else:
                boxes_t  = torch.zeros((0, 4), dtype=torch.float32)
                labels_t = torch.zeros((0,),   dtype=torch.int64)

            self.box_cache[img_info['id']] = (boxes_t, labels_t)

        cache_mb = sum(v.numel() for v in self.img_cache.values()) * 4 / 1e6
        print(f'Done. Cached {len(self.images)} images ({cache_mb:.0f} MB).')

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_info         = self.images[idx]
        img_t            = self.img_cache[img_info['id']].clone()
        boxes_t, labels_t = self.box_cache[img_info['id']]
        boxes_t          = boxes_t.clone()

        # Horizontal flip augmentation (training only)
        if self.augment and random.random() < 0.5:
            img_t   = TF.hflip(img_t)
            if boxes_t.shape[0] > 0:
                x1 = TARGET_SIZE - boxes_t[:, 2]
                x2 = TARGET_SIZE - boxes_t[:, 0]
                boxes_t = torch.stack([x1, boxes_t[:, 1],
                                       x2, boxes_t[:, 3]], dim=1)

        target = {
            'boxes'   : boxes_t,
            'labels'  : labels_t,
            'image_id': torch.tensor([img_info['id']]),
        }

        if self.transforms:
            img_t, target = self.transforms(img_t, target)

        return img_t, target


def collate_fn(batch):
    return tuple(zip(*batch))


def build_dataloaders(batch_size, num_workers=4):
    train_ds = OrangeDataset(TRAIN_DIR, augment=True)   # augment only train
    valid_ds = OrangeDataset(VALID_DIR, augment=False)
    test_ds  = OrangeDataset(TEST_DIR,  augment=False)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, collate_fn=collate_fn,
                              pin_memory=True, persistent_workers=(num_workers > 0))
    valid_loader = DataLoader(valid_ds, batch_size=1, shuffle=False,
                              num_workers=num_workers, collate_fn=collate_fn,
                              pin_memory=True, persistent_workers=(num_workers > 0))
    test_loader  = DataLoader(test_ds,  batch_size=1, shuffle=False,
                              num_workers=num_workers, collate_fn=collate_fn,
                              pin_memory=True, persistent_workers=(num_workers > 0))

    print(f'Train: {len(train_ds)} images  (augment=True)')
    print(f'Valid: {len(valid_ds)} images  (augment=False)')
    print(f'Test : {len(test_ds)} images  (augment=False)')
    return train_loader, valid_loader, test_loader

In [ ]:
# Cell 5: Build DataLoaders
train_loader, valid_loader, test_loader = build_dataloaders(
    batch_size  = CONFIG['batch_size'],
    num_workers = CONFIG['num_workers'],
)

In [ ]:
# Cell 6: Model builder

def build_faster_rcnn(backbone_key):
    """
    Returns a Faster R-CNN model with the head replaced for NUM_CLASSES.
    backbone_key must be one of: 'resnet50', 'resnet50v2', 'mobilenet'
    """
    if backbone_key not in MODEL_DETAILS:
        raise ValueError(f"Unknown backbone '{backbone_key}'. Choose from: {list(MODEL_DETAILS.keys())}")

    constructor, weights, display_name = MODEL_DETAILS[backbone_key]

    # Load with pretrained weights
    model = constructor(weights=weights)

    # Replace the classification head
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)

    model.to(DEVICE)
    print(f"Built {display_name} -> {NUM_CLASSES} classes (in_features={in_features})")
    return model

In [ ]:
# Cell 7: Training loop

def train_one_epoch(model, optimizer, loader, device, epoch):
    model.train()
    total_loss = 0.0
    scaler = torch.amp.GradScaler('cuda')

    for i, (images, targets) in enumerate(loader):
        images  = [img.to(device, non_blocking=True) for img in images]
        targets = [{k: v.to(device, non_blocking=True) for k, v in t.items()} for t in targets]

        with torch.amp.autocast('cuda'):
            loss_dict = model(images, targets)
            losses    = sum(loss_dict.values())

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(losses).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += losses.item()

        if (i + 1) % 20 == 0:
            print(f'  Epoch {epoch}  batch {i+1}/{len(loader)}  '
                  f'loss={losses.item():.4f}  '
                  f'(cls={loss_dict.get("loss_classifier",   torch.tensor(0)).item():.3f}  '
                  f'box={loss_dict.get("loss_box_reg",       torch.tensor(0)).item():.3f}  '
                  f'obj={loss_dict.get("loss_objectness",    torch.tensor(0)).item():.3f}  '
                  f'rpn={loss_dict.get("loss_rpn_box_reg",   torch.tensor(0)).item():.3f})')

    return total_loss / len(loader)


@torch.no_grad()
def evaluate_loss(model, loader, device):
    model.train()   # Faster R-CNN only returns losses in train mode
    total = 0.0
    for images, targets in loader:
        images  = [img.to(device, non_blocking=True) for img in images]
        targets = [{k: v.to(device, non_blocking=True) for k, v in t.items()} for t in targets]
        with torch.amp.autocast('cuda'):
            total += sum(model(images, targets).values()).item()
    return total / len(loader)


def run_model(backbone_key, train_loader, valid_loader):
    _, _, display_name = MODEL_DETAILS[backbone_key]
    run_dir  = os.path.join(CONFIG['save_path'], backbone_key)
    os.makedirs(run_dir, exist_ok=True)

    model  = build_faster_rcnn(backbone_key)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(
        params, lr=CONFIG['lr'], momentum=CONFIG['momentum'],
        weight_decay=CONFIG['weight_decay'],
    )
    lr_scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=CONFIG['lr_step_size'], gamma=CONFIG['lr_gamma'],
    )

    history   = {'train_loss': [], 'val_loss': []}
    best_val  = float('inf')
    best_path = os.path.join(run_dir, 'best.pth')


    print(f'Training {display_name}  for {CONFIG["epochs"]} epochs')
    print(f'Run dir: {run_dir}')

    for epoch in range(1, CONFIG['epochs'] + 1):
        t0         = time.time()
        train_loss = train_one_epoch(model, optimizer, train_loader, DEVICE, epoch)
        val_loss   = evaluate_loss(model, valid_loader, DEVICE)
        lr_scheduler.step()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)

        print(f'Epoch {epoch:>3}/{CONFIG["epochs"]}  '
              f'train={train_loss:.4f}  val={val_loss:.4f}  '
              f'({time.time()-t0:.1f}s)')

        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), best_path)
            print(f'  ↳ New best val_loss={best_val:.4f} — checkpoint saved')

    # Save final weights to Drive
    torch.save(model.state_dict(), os.path.join(run_dir, 'final.pth'))

    # Save history to Drive
    with open(os.path.join(run_dir, 'history.json'), 'w') as f:
        json.dump({'train_loss': history['train_loss'],
                   'val_loss':   history['val_loss'],
                   'best_val_loss': best_val}, f, indent=2)

    # Save loss curve to Drive
    plt.figure(figsize=(8, 4))
    plt.plot(history['train_loss'], label='Train')
    plt.plot(history['val_loss'],   label='Val')
    plt.xlabel('Epoch'); plt.ylabel('Loss')
    plt.title(f'Loss Curves — {display_name}')
    plt.legend(); plt.tight_layout()
    plt.savefig(os.path.join(run_dir, 'loss_curves.png'), dpi=120)
    plt.show()

    return model, history, run_dir, best_path


print('Training functions defined.')

In [ ]:
# Cell 8: Evaluation / Metrics

def compute_iou(box_a, box_b):
    xA = max(box_a[0], box_b[0]); yA = max(box_a[1], box_b[1])
    xB = min(box_a[2], box_b[2]); yB = min(box_a[3], box_b[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = (box_a[2]-box_a[0]) * (box_a[3]-box_a[1])
    areaB = (box_b[2]-box_b[0]) * (box_b[3]-box_b[1])
    union = areaA + areaB - inter
    return inter / union if union > 0 else 0.0


def match_predictions(pred_boxes, pred_scores, gt_boxes, iou_threshold=0.5):
    if len(gt_boxes) == 0:
        return 0, len(pred_boxes), 0
    if len(pred_boxes) == 0:
        return 0, 0, len(gt_boxes)

    order   = sorted(range(len(pred_scores)), key=lambda i: pred_scores[i], reverse=True)
    matched = [False] * len(gt_boxes)
    tp = fp = 0

    for pi in order:
        best_iou, best_gi = 0.0, -1
        for gi, gb in enumerate(gt_boxes):
            if matched[gi]: continue
            iou = compute_iou(pred_boxes[pi], gb)
            if iou > best_iou: best_iou, best_gi = iou, gi
        if best_iou >= iou_threshold:
            tp += 1; matched[best_gi] = True
        else:
            fp += 1

    return tp, fp, matched.count(False)


def compute_ap(precisions, recalls):
    return sum(max((p for p, r in zip(precisions, recalls) if r >= t), default=0.0)
               for t in np.linspace(0, 1, 11)) / 11


@torch.no_grad()
def eval_model(backbone_key, best_path, test_loader, conf=0.5, iou_threshold=0.5):
    _, _, display_name = MODEL_DETAILS[backbone_key]
    model = build_faster_rcnn(backbone_key)
    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    model.eval()

    all_tp = all_fp = all_fn = 0
    count_errors = []
    all_scores, all_is_tp = [], []
    total_gt = 0

    for images, targets in test_loader:
        images = [img.to(DEVICE) for img in images]
        preds  = model(images)

        for pred, target in zip(preds, targets):
            gt_boxes   = target['boxes'].tolist()
            pred_boxes = pred['boxes'].tolist()
            scores     = pred['scores'].tolist()

            keep       = [i for i, s in enumerate(scores) if s >= conf]
            pred_boxes = [pred_boxes[i] for i in keep]
            scores     = [scores[i]     for i in keep]

            tp, fp, fn = match_predictions(pred_boxes, scores, gt_boxes, iou_threshold)
            all_tp += tp; all_fp += fp; all_fn += fn
            count_errors.append(len(pred_boxes) - len(gt_boxes))
            total_gt += len(gt_boxes)

            matched = [False] * len(gt_boxes)
            for pi in sorted(range(len(scores)), key=lambda i: scores[i], reverse=True):
                best_iou, best_gi = 0.0, -1
                for gi, gb in enumerate(gt_boxes):
                    if matched[gi]: continue
                    iou = compute_iou(pred_boxes[pi], gb)
                    if iou > best_iou: best_iou, best_gi = iou, gi
                if best_iou >= iou_threshold:
                    all_is_tp.append(1); matched[best_gi] = True
                else:
                    all_is_tp.append(0)
                all_scores.append(scores[pi])

    precision = all_tp / (all_tp + all_fp + 1e-8)
    recall    = all_tp / (all_tp + all_fn + 1e-8)
    f1        = 2 * precision * recall / (precision + recall + 1e-8)

    order = sorted(range(len(all_scores)), key=lambda i: all_scores[i], reverse=True)
    tp_cum = fp_cum = 0
    prec_list, rec_list = [], []
    for i in order:
        if all_is_tp[i]: tp_cum += 1
        else:            fp_cum += 1
        prec_list.append(tp_cum / (tp_cum + fp_cum))
        rec_list.append(tp_cum  / (total_gt + 1e-8))
    ap = compute_ap(prec_list, rec_list)

    errors = np.array(count_errors, dtype=float)
    mae    = np.mean(np.abs(errors))
    rmse   = np.sqrt(np.mean(errors ** 2))

    metrics = {
        'backbone' : backbone_key,
        'precision': round(precision, 4),
        'recall'   : round(recall,    4),
        'f1'       : round(f1,        4),
        'mAP@0.5'  : round(ap,        4),
        'MAE'      : round(mae,        4),
        'RMSE'     : round(rmse,       4),
        'TP': all_tp, 'FP': all_fp, 'FN': all_fn,
    }


    print(f'Results — {display_name}  (conf={conf})')
    for k, v in metrics.items():
        print(f'  {k:<12}: {v}')

    # Save metrics to Drive
    with open(os.path.join(os.path.dirname(best_path), 'metrics.json'), 'w') as f:
        json.dump({k: (float(v) if isinstance(v, (np.integer, np.floating)) else v)
                   for k, v in metrics.items()}, f, indent=2)

    return metrics

In [ ]:
# Cell 9: Confidence threshold sweep

@torch.no_grad()
def sweep_conf_threshold(backbone_key, best_path, valid_loader, thresholds=None):
    if thresholds is None:
        thresholds = [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9]

    _, _, display_name = MODEL_DETAILS[backbone_key]
    model = build_faster_rcnn(backbone_key)
    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    model.eval()

    results = []
    for conf in thresholds:
        errors = []
        for images, targets in valid_loader:
            images = [img.to(DEVICE) for img in images]
            preds  = model(images)
            for pred, target in zip(preds, targets):
                gt_count   = len(target['boxes'])
                pred_count = sum(1 for s in pred['scores'].tolist() if s >= conf)
                errors.append(pred_count - gt_count)

        arr  = np.array(errors, dtype=float)
        mae  = np.mean(np.abs(arr))
        rmse = np.sqrt(np.mean(arr ** 2))
        results.append({'conf': conf, 'mae': mae, 'rmse': rmse})
        print(f'  conf={conf:.2f}  MAE={mae:.3f}  RMSE={rmse:.3f}')

    df       = pd.DataFrame(results)
    best_row = df.loc[df['mae'].idxmin()]
    print(f'\nBest threshold for {display_name}: {best_row["conf"]}  (MAE={best_row["mae"]:.3f})')

    plt.figure(figsize=(7, 4))
    plt.plot(df['conf'], df['mae'],  marker='o', label='MAE')
    plt.plot(df['conf'], df['rmse'], marker='s', label='RMSE')
    plt.axvline(best_row['conf'], color='red', linestyle='--',
                label=f'best conf={best_row["conf"]}')
    plt.xlabel('Confidence threshold'); plt.ylabel('Error')
    plt.title(f'Conf Sweep — {display_name}')
    plt.legend(); plt.tight_layout()

    # Save to Drive
    sweep_dir = os.path.dirname(best_path)
    df.to_csv(os.path.join(sweep_dir, 'conf_sweep.csv'), index=False)
    plt.savefig(os.path.join(sweep_dir, 'conf_sweep.png'), dpi=120)
    plt.show()

    return df, float(best_row['conf'])

In [ ]:
# Cell 10: Full backbone sweep

all_rows = []

for backbone_key in MODEL_DETAILS:
    start = datetime.now()
    model, history, run_dir, best_path = run_model(
        backbone_key, train_loader, valid_loader
    )
    end = datetime.now()

    _, best_conf = sweep_conf_threshold(backbone_key, best_path, valid_loader)

    metrics = eval_model(
        backbone_key = backbone_key,
        best_path    = best_path,
        test_loader  = test_loader,
        conf         = best_conf,
    )

    all_rows.append({
        'backbone'  : backbone_key,
        'display'   : MODEL_DETAILS[backbone_key][2],
        'run_dir'   : run_dir,
        'best_path' : best_path,
        'best_conf' : best_conf,
        'train_time': (end - start),
        **metrics,
    })

results_df = pd.DataFrame(all_rows)
results_df.to_csv(os.path.join(CONFIG['save_path'], 'summary.csv'), index=False)
print(f'Summary saved: {CONFIG["save_path"]}/summary.csv')
results_df

In [ ]:
# Cell 11: Results summary + bar charts

display_cols = ["display", "precision", "recall", "f1", "mAP@0.5", "MAE", "RMSE", "train_time"]
print(results_df[display_cols].to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
labels = results_df["display"].tolist()

axes[0].bar(labels, results_df["mAP@0.5"], color='steelblue')
axes[0].set_title("mAP @ IoU 0.5"); axes[0].set_ylabel("mAP")
axes[0].set_ylim(0, 1); axes[0].tick_params(axis='x', rotation=15)

axes[1].bar(labels, results_df["MAE"], color='coral')
axes[1].set_title("MAE (count error)"); axes[1].set_ylabel("MAE")
axes[1].tick_params(axis='x', rotation=15)

axes[2].bar(labels, results_df["f1"], color='mediumseagreen')
axes[2].set_title("F1 Score"); axes[2].set_ylabel("F1")
axes[2].set_ylim(0, 1); axes[2].tick_params(axis='x', rotation=15)

plt.suptitle("Faster R-CNN Backbone Comparison — Orange Detection", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['save_path'], "backbone_comparison.png"), dpi=120)
plt.show()

In [ ]:
# Cell 12: Qualitative visualisation: inference on test images

@torch.no_grad()
def visualise_predictions(backbone_key, best_path, conf=0.5, n=4):
    _, _, display_name = MODEL_DETAILS[backbone_key]
    model = build_faster_rcnn(backbone_key)
    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    model.eval()

    ds   = OrangeDataset(TEST_DIR)   # just TEST_DIR directly
    idxs = random.sample(range(len(ds)), min(n, len(ds)))

    fig, axes = plt.subplots(1, len(idxs), figsize=(6 * len(idxs), 6))
    if len(idxs) == 1: axes = [axes]

    for ax, idx in zip(axes, idxs):
        img_t, target = ds[idx]
        pred = model([img_t.to(DEVICE)])[0]

        ax.imshow(img_t.permute(1, 2, 0).numpy())

        for box in target['boxes']:
            x1, y1, x2, y2 = box.tolist()
            ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                           lw=2, edgecolor='lime', facecolor='none'))

        for box, score in zip(pred['boxes'].tolist(), pred['scores'].tolist()):
            if score < conf: continue
            x1, y1, x2, y2 = box
            ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                           lw=2, edgecolor='orange', facecolor='none'))
            ax.text(x1, y1-4, f'{score:.2f}', color='orange', fontsize=8,
                    bbox=dict(facecolor='black', alpha=0.4, pad=1))

        gt_n   = len(target['boxes'])
        pred_n = sum(1 for s in pred['scores'].tolist() if s >= conf)
        ax.set_title(f'GT: {gt_n}  |  Pred: {pred_n}\n(green=GT, orange=pred)')
        ax.axis('off')

    plt.suptitle(f'{display_name} — test-set predictions', fontsize=12)
    plt.tight_layout()
    viz_dir = os.path.join(CONFIG['save_path'], backbone_key)
    plt.savefig(os.path.join(viz_dir, 'test_predictions.png'), dpi=120, bbox_inches='tight')
    plt.show()


# Visualise each backbone using its own saved checkpoint
for bk in MODEL_DETAILS:
    visualise_predictions(
        backbone_key = bk,
        best_path    = os.path.join(CONFIG['save_path'], bk, 'best.pth'),
        conf         = 0.5,
        n            = 4,
    )